In [1]:
import pandas as pd
from os import listdir
from os.path import isfile, join
import re

In [2]:
plate_template = pd.read_excel("Plate Layout.xlsx", sheet_name = 0, header=None)
plate_template

,0,1,2,3,4,5,6,7,8,9,10,11
0,P1-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,P2-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,P1-2,1.0,NaN,6.0,NaN,8.0,NaN,11.0,NaN,3.0,NaN,0
3,P2-2,7.0,NaN,5.0,NaN,12.0,NaN,10.0,NaN,10.0,NaN,0
4,P1-3,NaN,8.0,NaN,1.0,NaN,7.0,NaN,11.0,NaN,9.0,0
5,P2-3,NaN,6.0,NaN,2.0,NaN,11.0,NaN,4.0,NaN,8.0,0
6,P1-4,3.0,NaN,2.0,NaN,12.0,NaN,11.0,NaN,10.0,NaN,0
7,P2-4,2.0,NaN,5.0,NaN,12.0,NaN,7.0,NaN,12.0,NaN,0
8,P1-5,NaN,7.0,NaN,12.0,NaN,6.0,NaN,9.0,NaN,2.0,0
9,P2-5,NaN,5.0,NaN,1.0,NaN,3.0,NaN,4.0,NaN,4.0,0


In [3]:
def get_abs(abs_df, absorbance, line):
    '''
    Function to extract the particular absorbance of a certain cell line and return that as a list
    
    Variables:
        abs_df = the pandas dataframe that contains the absorbances of all cell lines
        plate_template = the positioning of the lines on each of the cell plates
        absorbance = well...pretty self explainatory 
        line = cell line that you're intersted in 
    '''
    
    #Reformat the dataframe
    abs_df = abs_df[(abs_df[14].str.contains("Blank")==False) & (abs_df[14].str.contains(str(absorbance))==True)].drop([1, 14], axis=1)
    abs_df = abs_df.reset_index()
    abs_df = abs_df.drop("index", axis=1)
    abs_df.columns = plate_template.columns

    #Select the indexes with the lines
    values = abs_df.where(plate_template == line).apply(lambda x: x.dropna().to_list())
    abs_list = sum(values[values.str.len() > 0].to_list(), [])

    if len(abs_list) != 5:
        print("AHHHHHHHHHHHHHHH", len(abs_list))
    
    return abs_list

# Fitness Assay Sets

## For FA 1

## For FA 4

In [4]:
data_list = []
absorb = 680
week_of_choice = "./Feb12_16/"

for f in listdir(week_of_choice):
    if "HE" not in f:
        continue
        
    f = week_of_choice +"/"+ f
    print(f)
    
    #Format of the excels:
    #Every sheet is a different condition. And they follow this order:
        #20@20
        #20@37
        #37@20
        #37@37
        #P@20
        #P@37
    cell_lines = ["20S","20S","37S","37S","P","P"]
    for index in range(0, 6, 2): #go through every sheet (condition) in the file
        #Get the date, time and cell line from the experiment file path 
        file = pd.read_excel(f, usecols = "A:B", nrows = 6, sheet_name = index, header=None)
        file_data = file[file[0].str.contains("Experiment File Path")==True].iloc[0,1]
        search = re.search(r".*\\FA(\d)\\([0-9]{4}-[0-9]{2}-[0-9]{2}[_-][0-9]{1,2}[APpa][mM])", file_data)
        FA = search.group(1)
        date = search.group(2)
        cell_line = cell_lines[index]
        
        print(date, cell_line)
        
        #Read the 20°C condition
        abs_df_20 = pd.read_excel(f, usecols = "B:O", skiprows = 34, nrows = 32, header = None, sheet_name = index)
        #print(abs_df_20.head())
        #Read the 37°C condition
        abs_df_37 = pd.read_excel(f, usecols = "B:O", skiprows = 34, nrows = 32, header = None, sheet_name = index+1)
        
        for colony in range(1,13):
            
            #Get the colony number
            #Get the absorbance
            
            abs_data_37 = get_abs(abs_df_37, absorb, colony)
            abs_data_20 = get_abs(abs_df_20, absorb, colony)
            
            data_list.append([date, FA, cell_line, 37, colony, abs_data_37])
            data_list.append([date, FA, cell_line, 20, colony, abs_data_20])

./Feb12_16//HE_FA2_240212.xlsm
2024-02-12_10AM 20S
2024-02-12_10AM 37S
2024-02-12_10AM P
./Feb12_16//HE_FA_240216 copy.xlsm
2024-02-14-6pm 20S
2024-02-14-6pm 37S
2024-02-14-6pm P
./Feb12_16//HE_FA_240216.xlsm
2024-02-16-10AM 20S
2024-02-16-10AM 37S
2024-02-16-10AM P
./Feb12_16//HE_FA_240215.xlsm
2024-02-15-6PM 20S
2024-02-15-6PM 37S
2024-02-15-6PM P
./Feb12_16//HE_FA_240214.xlsm
2024-02-14-10AM 20S
2024-02-14-10AM 37S
2024-02-14-10AM P
./Feb12_16//HE_FA_240213 copy.xlsm
2024-02-13_6PM 20S
2024-02-13_6PM 37S
2024-02-13_6PM P
./Feb12_16//HE_FA_240216 - Copy.xlsm
2024-02-16-6PM 20S
2024-02-16-6PM 37S
2024-02-16-6PM P
./Feb12_16//HE_FA_240213.xlsm
2024-02-13-10AM 20S
2024-02-13-10AM 37S
2024-02-13-10AM P
./Feb12_16//HE_FA_240215 copy.xlsm
2024-02-15_10AM 20S
2024-02-15_10AM 37S
2024-02-15_10AM P
./Feb12_16//HE_FA_240212.xlsm
2024-02-12_6PM 20S
2024-02-12_6PM 37S
2024-02-12_6PM P


# FA 5+

In [5]:
listdir(week_of_choice)

['HE_FA2_240212.xlsm',
 '.DS_Store',
 'HE_FA_240216 copy.xlsm',
 'HE_FA_240216.xlsm',
 'HE_FA_240215.xlsm',
 'HE_FA_240214.xlsm',
 'HE_FA_240213 copy.xlsm',
 'HE_FA_240216 - Copy.xlsm',
 'HE_FA_240213.xlsm',
 'HE_FA_240215 copy.xlsm',
 'HE_FA_240212.xlsm']

In [11]:
data_list = []
absorb = 680
week_of_choice = "./Aug19_23"

#If have to rewrite the plate format
if "Plate Layout.xlsx" in listdir(week_of_choice):
    plate_template = pd.read_excel(week_of_choice +"/"+"Plate Layout.xlsx", sheet_name = 0, header=None)

for f in listdir(week_of_choice):
    if "HE" not in f:
        continue
        
    f = week_of_choice +"/"+ f
    print(f)
    
    #Format of the excels:
    #Every sheet is a different condition. And they follow this order:
        #20@20
        #20@37
        #37@20
        #37@37
        #P@20
        #P@37
    cell_lines = ["20S","20S","37S","37S","P","P", "25S", "25S"]
    for index in range(0, 8, 2): #go through every sheet (condition) in the file
        #Get the date, time and cell line from the experiment file path 
        file = pd.read_excel(f, usecols = "A:B", nrows = 6, sheet_name = index, header=None)
        file_data = file[file[0].str.contains("Experiment File Path")==True].iloc[0,1]
        search = re.search(r".*\\FA(\d+)\\([0-9]{4}-[0-9]{2}-[0-9]{2}[_-][0-9]{1,2}[APpa][mM])", file_data)
        FA = search.group(1)
        date = search.group(2)
        cell_line = cell_lines[index]
        
        print(date, cell_line)
        
        #Read the 20°C condition
        abs_df_20 = pd.read_excel(f, usecols = "B:O", skiprows = 34, nrows = 32, header = None, sheet_name = index)
        #print(abs_df_20.head())
        #Read the 37°C condition
        abs_df_37 = pd.read_excel(f, usecols = "B:O", skiprows = 34, nrows = 32, header = None, sheet_name = index+1)
        
        for colony in range(1,13):
            
            #Get the colony number
            #Get the absorbance
            
            abs_data_37 = get_abs(abs_df_37, absorb, colony)
            abs_data_20 = get_abs(abs_df_20, absorb, colony)
            
            data_list.append([date, FA, cell_line, 37, colony, abs_data_37])
            data_list.append([date, FA, cell_line, 20, colony, abs_data_20])

./Aug19_23/HE_FA_240821.xlsm
2024-08-21_10AM 20S
2024-08-21_10AM 37S
2024-08-21_10AM P
2024-08-21_10AM 25S
./Aug19_23/HE_FA_240820.xlsm
2024-08-20_10AM 20S
2024-08-20_10AM 37S
2024-08-20_10AM P
2024-08-20_10AM 25S
./Aug19_23/HE_FA_240822 2.xlsm
2024-08-22_6PM 20S
2024-08-22_6PM 37S
2024-08-22_6PM P
2024-08-22_6PM 25S
./Aug19_23/HE_FA_240820 2.xlsm
2024-08-20_6PM 20S
2024-08-20_6PM 37S
2024-08-20_6PM P
2024-08-20_6PM 25S
./Aug19_23/HE_FA_240819.xlsm
2024-08-19_10AM 20S
2024-08-19_10AM 37S
2024-08-19_10AM P
2024-08-19_10AM 25S
./Aug19_23/HE_FA_240823.xlsm
2024-08-23_10AM 20S
2024-08-23_10AM 37S
2024-08-23_10AM P
2024-08-23_10AM 25S
./Aug19_23/HE_FA_240819 2.xlsm
2024-08-19_6PM 20S
2024-08-19_6PM 37S
2024-08-19_6PM P
2024-08-19_6PM 25S
./Aug19_23/HE_FA_240821 2.xlsm
2024-08-21_6PM 20S
2024-08-21_6PM 37S
2024-08-21_6PM P
2024-08-21_6PM 25S
./Aug19_23/HE_FA_240822.xlsm
2024-08-22_10AM 20S
2024-08-22_10AM 37S
2024-08-22_10AM P
2024-08-22_10AM 25S


# Cells

In [14]:
data_list

df = pd.DataFrame(data_list, columns =['Date', 'FA#', 'Condition', "Temperature", "Colony", "Absorbance"])
df.head()

,Date,FA#,Condition,Temperature,Colony,Absorbance
0,2024-08-21_10AM,10,20S,37,1,"[0.907, 0.605, 0.893, 0.899, 0.941]"
1,2024-08-21_10AM,10,20S,20,1,"[1.092, 1.015, 1.051, 0.998, 0.808]"
2,2024-08-21_10AM,10,20S,37,2,"[0.972, 0.892, 0.989, 0.958, 1.06]"
3,2024-08-21_10AM,10,20S,20,2,"[1.068, 1.143, 1.31, 1.234, 1.182]"
4,2024-08-21_10AM,10,20S,37,3,"[0.911, 0.912, 0.932, 0.96, 0.942]"


In [15]:
df["Absorbance"]

0      [0.907, 0.605, 0.893, 0.899, 0.941]
1      [1.092, 1.015, 1.051, 0.998, 0.808]
2       [0.972, 0.892, 0.989, 0.958, 1.06]
3       [1.068, 1.143, 1.31, 1.234, 1.182]
4       [0.911, 0.912, 0.932, 0.96, 0.942]
                      ...                 
859     [0.966, 1.27, 1.023, 1.307, 0.962]
860    [0.764, 0.772, 0.803, 0.822, 0.777]
861    [0.716, 0.876, 0.871, 0.801, 0.951]
862    [0.932, 0.909, 0.922, 0.774, 0.837]
863    [0.862, 1.093, 0.848, 0.941, 1.003]
Name: Absorbance, Length: 864, dtype: object

In [16]:
# remove the spacer between the time and the date
df["Date"]=df["Date"].str.upper().str.slice_replace(10,11,' ')


In [17]:
df["Date"] = pd.to_datetime(df["Date"])
df[['Rep_1', 'Rep_2', 'Rep_3', 'Rep_4', 'Rep_5']]  = pd.DataFrame(df.Absorbance.tolist())
print(df.shape)
df.head()


(864, 11)


,Date,FA#,Condition,Temperature,Colony,Absorbance,Rep_1,Rep_2,Rep_3,Rep_4,Rep_5
0,2024-08-21 10:00:00,10,20S,37,1,"[0.907, 0.605, 0.893, 0.899, 0.941]",0.907,0.605,0.893,0.899,0.941
1,2024-08-21 10:00:00,10,20S,20,1,"[1.092, 1.015, 1.051, 0.998, 0.808]",1.092,1.015,1.051,0.998,0.808
2,2024-08-21 10:00:00,10,20S,37,2,"[0.972, 0.892, 0.989, 0.958, 1.06]",0.972,0.892,0.989,0.958,1.060
3,2024-08-21 10:00:00,10,20S,20,2,"[1.068, 1.143, 1.31, 1.234, 1.182]",1.068,1.143,1.310,1.234,1.182
4,2024-08-21 10:00:00,10,20S,37,3,"[0.911, 0.912, 0.932, 0.96, 0.942]",0.911,0.912,0.932,0.960,0.942


In [18]:
"FA_parsed/FA{FA_num}.csv".format(FA_num = FA)

'FA_parsed/FA10.csv'

In [24]:
df_sub = df.drop(['Absorbance'], axis=1)
df_sub.to_csv("../Data_Cleaned/FA{FA_num}.csv".format(FA_num = FA), index=False)

In [20]:
df_sub.head()

,Date,FA#,Condition,Temperature,Colony,Rep_1,Rep_2,Rep_3,Rep_4,Rep_5
0,2024-08-21 10:00:00,10,20S,37,1,0.907,0.605,0.893,0.899,0.941
1,2024-08-21 10:00:00,10,20S,20,1,1.092,1.015,1.051,0.998,0.808
2,2024-08-21 10:00:00,10,20S,37,2,0.972,0.892,0.989,0.958,1.060
3,2024-08-21 10:00:00,10,20S,20,2,1.068,1.143,1.310,1.234,1.182
4,2024-08-21 10:00:00,10,20S,37,3,0.911,0.912,0.932,0.960,0.942


In [21]:
df_sub.shape

(864, 10)

In [22]:
df_sub[df_sub.Colony == 8]

,Date,FA#,Condition,Temperature,Colony,Rep_1,Rep_2,Rep_3,Rep_4,Rep_5
14,2024-08-21 10:00:00,10,20S,37,8,0.711,0.767,0.731,0.749,0.801
15,2024-08-21 10:00:00,10,20S,20,8,1.363,1.189,1.143,1.236,1.074
38,2024-08-21 10:00:00,10,37S,37,8,1.057,0.955,1.021,0.863,0.883
39,2024-08-21 10:00:00,10,37S,20,8,0.833,0.899,0.813,0.819,0.886
62,2024-08-21 10:00:00,10,P,37,8,1.224,1.207,1.232,1.083,1.212
...,...,...,...,...,...,...,...,...,...,...
807,2024-08-22 10:00:00,10,37S,20,8,0.896,0.884,0.943,0.893,0.828
830,2024-08-22 10:00:00,10,P,37,8,1.218,1.221,1.244,1.143,1.226
831,2024-08-22 10:00:00,10,P,20,8,1.077,1.005,0.037,0.036,0.036
854,2024-08-22 10:00:00,10,25S,37,8,1.172,1.175,0.949,1.018,1.009
